In [1]:
import xgboost 
import pickle
import pandas as pd

# load the pickled object safely from file
with open("best_features_arr.pkl", "rb") as _f:
    arr_cats = pickle.load(_f)
with open('best_features_ven.pkl', 'rb') as _f:
    ven_cats = pickle.load(_f)
with open('xgb_model_arr_med.pkl', 'rb') as _f:
    xgb_model_arr_med = pickle.load(_f)
with open('xgb_model_ven_med.pkl', 'rb') as _f:
    xgb_model_ven_med = pickle.load(_f)
with open('list_barrios.pkl', 'rb') as _f:
    list_barrios = pickle.load(_f)
with open('price_per_m2_ven.pkl', 'rb') as _f:
    ppmc_ven = pickle.load(_f)
with open('price_per_space_ven.pkl', 'rb') as _f:
    pppz_ven = pickle.load(_f)
with open('price_per_m2_arr.pkl', 'rb') as _f:
    ppmc_arr = pickle.load(_f)
with open('price_per_space_arr.pkl', 'rb') as _f:
    pppz_arr = pickle.load(_f)
with open('preprocessor.pkl', 'rb') as _f:
    preprocessor = pickle.load(_f)
loan = pd.read_csv('arr_mede_final.csv')
sales = pd.read_csv('ven_mede_final.csv')
def pred_func_arr(area, habitaciones, banos, parqueaderos, barrio, tipo):
    """
    Make prediction for rental price using the trained pipeline
    """
    # Normalizar y comprobar barrio (case-insensitive)
    barrio_norm = str(barrio).strip().lower()
    barrios_lc = [b.strip().lower() for b in list_barrios]
    if barrio_norm not in barrios_lc:
        return f'El barrio \"{barrio}\" no está en la lista de barrios conocidos. Por favor, elija uno de los siguientes: {list_barrios}'
    tipo_norm = str(tipo).strip().lower()
    tipos_lc = [t.strip().lower() for t in loan['tipo'].unique().tolist()]
    if tipo_norm not in tipos_lc:
        return f'El tipo \"{tipo}\" no está en la lista de tipos conocidos. Por favor, elija uno de los siguientes: {loan["tipo"].unique().tolist()}'
    # Crear DataFrame de entrada con las columnas esperadas por el preprocessor
    cols = arr_cats
    input_df = pd.DataFrame([{
        'habitaciones': habitaciones,
        'baños': banos,
        'parqueaderos': parqueaderos,
        'espacios': None,  # se calculará abajo
        'axe': None,
        'tipo': tipo,
        'ppmc': None,
        'pppz': None,
        'garaje_bin': None,
        'parqueadero2': None,
        'new_index': None,
        'area': area,
        'barrio': barrio,
        'axh': None,
        'axa': None
    }])
    
    # Calcular espacios y axe (asegurando no división por cero)
    input_df['espacios'] = input_df['habitaciones'] + input_df['parqueaderos'] + input_df['baños']
    input_df['axe'] = input_df['area'] / input_df['espacios'].replace({0: pd.NA})
    input_df['axh'] = input_df['area'] / input_df['habitaciones'].replace({0: pd.NA})
    input_df['axa'] = input_df['area'] * input_df['area']
    input_df['parqueadero2'] = input_df['parqueaderos'] * input_df['parqueaderos']
    input_df['ppmc'] = input_df['barrio'].map(ppmc_arr)
    input_df['pppz'] = input_df['barrio'].map(pppz_arr)
    input_df['new_index'] = input_df['ppmc']/(input_df['ppmc'].max())*100
    input_df['garaje_bin'] = input_df['parqueaderos'].apply(lambda x: 1 if x > 0 else 0)
    
    input_df = input_df[cols]
   

    # Transformar datos de entrada con el preprocessor ya entrenado
    input_transformed = preprocessor.transform(input_df)

    # Si es matriz dispersa, convertir a densa
    if hasattr(input_transformed, "toarray"):
        input_transformed = input_transformed.toarray()

    pred = xgb_model_arr_med.predict(input_transformed)[0]
    return pred 

def pred_func_ven(area, habitaciones, banos, parqueaderos, barrio, tipo):
    """
    Make prediction for rental price using the trained pipeline
    """
    # Normalizar y comprobar barrio (case-insensitive)
    barrio_norm = str(barrio).strip().lower()
    barrios_lc = [b.strip().lower() for b in list_barrios]
    if barrio_norm not in barrios_lc:
        return f'El barrio \"{barrio}\" no está en la lista de barrios conocidos. Por favor, elija uno de los siguientes: {list_barrios}'
    tipo_norm = str(tipo).strip().lower()
    tipos_lc = [t.strip().lower() for t in sales['tipo'].unique().tolist()]
    if tipo_norm not in tipos_lc:
        return f'El tipo \"{tipo}\" no está en la lista de tipos conocidos. Por favor, elija uno de los siguientes: {sales["tipo"].unique().tolist()}'
    # Crear DataFrame de entrada con las columnas esperadas por el preprocessor
    cols = ven_cats
    input_df = pd.DataFrame([{
        'habitaciones': habitaciones,
        'baños': banos,
        'parqueaderos': parqueaderos,
        'espacios': None,  # se calculará abajo
        'axe': None,
        'tipo': tipo,
        'ppmc': None,
        'pppz': None,
        'garaje_bin': None,
        'parqueadero2': None,
        'new_index': None,
        'area': area,
        'barrio': barrio,
        'axh': None,
        'axa': None
    }])
    
    # Calcular espacios y axe (asegurando no división por cero)
    input_df['espacios'] = input_df['habitaciones'] + input_df['parqueaderos'] + input_df['baños']
    input_df['axe'] = input_df['area'] / input_df['espacios'].replace({0: pd.NA})
    input_df['axh'] = input_df['area'] / input_df['habitaciones'].replace({0: pd.NA})
    input_df['axa'] = input_df['area'] * input_df['area']
    input_df['parqueadero2'] = input_df['parqueaderos'] * input_df['parqueaderos']
    input_df['ppmc'] = input_df['barrio'].map(ppmc_arr)
    input_df['pppz'] = input_df['barrio'].map(pppz_arr)
    input_df['new_index'] = input_df['ppmc']/(input_df['ppmc'].max())*100
    input_df['garaje_bin'] = input_df['parqueaderos'].apply(lambda x: 1 if x > 0 else 0)
    
    input_df = input_df[cols]
   

    # Transformar datos de entrada con el preprocessor ya entrenado
    input_transformed = preprocessor.transform(input_df)

    # Si es matriz dispersa, convertir a densa
    if hasattr(input_transformed, "toarray"):
        input_transformed = input_transformed.toarray()

    pred = xgb_model_ven_med.predict(input_transformed)[0]
    return pred 


In [2]:
pred_func_ven(100, 3, 2, 1, 'El Poblado', 'Apartamento')

KeyError: "None of [Index(['num__parqueaderos', 'num__espacios', 'num__pppz', 'num__habitaciones',\n       'num__parq2', 'cat__tipo_casa'],\n      dtype='object')] are in the [columns]"